# Aula 14 — Menor Caminho: Dijkstra e Roteamento Dinâmico

**UNIFEI • ECAA08 Automática • Grupo 2**
**Projeto:** SCADA da Máquina Pneumática de Envasamento de Água em Copos.

## 1. Objetivo e vínculo com o projeto

Calcular a menor distância entre equipamentos da rede de abastecimento, recalculando a rota após bloqueio de equipamento, isolamento de tubulação ou alteração de comprimento. O entregável é a classe `DijkstraRouter`, implementada integralmente no notebook de mesmo nome desta aula.

A rede preserva os nomes e os nove trechos do modelo das Aulas 11 e 15. Bombas redundantes e comprimentos pertencem àquela abstração didática; não constituem levantamento físico validado. O resultado é uma **proposta de rota**. Nenhum comando é enviado ao Automation Studio, às válvulas ou aos cilindros. O sequenciamento e os permissivos continuam pertencendo ao controle da máquina.

| Origem | Destino | Comprimento (cm) |
|---|---|---:|
| TK-101_Agua | P-101_BombaA | 150 |
| TK-101_Agua | P-102_BombaB | 150 |
| P-101_BombaA | MAN-200_Agua | 80 |
| P-102_BombaB | MAN-200_Agua | 85 |
| MAN-200_Agua | DOS-201_Dosador | 40 |
| DOS-201_Dosador | BIC-202_Bico | 20 |
| BIC-202_Bico | MESA-000_Index | 10 |
| MAG-100_Copos | MESA-000_Index | 15 |
| MESA-000_Index | EST-500_Saida | 50 |

O cadastro completo permite comparar com as aulas anteriores. A consulta hidráulica principal termina no dosador, evitando interpretar o transporte do copo como tubulação. `DOS-201_Dosador` representa o cilindro C; `BIC-202_Bico`, o bico; `MESA-000_Index`, a mesa movimentada por M-001; `EST-500_Saida`, a esteira M-002. São identificadores topológicos herdados, não novas tags de instrumentação.

```mermaid
flowchart LR
    T[TK-101 Água] -->|150 cm| A[P-101 Bomba A]
    T -->|150 cm| B[P-102 Bomba B]
    A -->|80 cm| M[MAN-200 Água]
    B -->|85 cm| M
    M -->|40 cm| D[DOS-201 Dosador]
    D -->|20 cm| N[BIC-202 Bico]
```

## 2. Formulação matemática

Para um dígrafo $G=(V,E,w)$ com $w:E\to\mathbb{R}_{\geq0}$, procuramos um caminho $P$ de $s$ até $t$ que minimize:

$$C(P)=\sum_{(u,v)\in P}w(u,v).$$

Inicializamos $d(s)=0$, $d(v)=\infty$ para os demais vértices. Ao retirar do heap o vértice com menor distância provisória, relaxamos suas arestas:

$$d(u)+w(u,v)<d(v)\Rightarrow d(v)\leftarrow d(u)+w(u,v),\quad pred(v)\leftarrow u.$$

Os predecessores permitem reconstruir a sequência final de equipamentos. Entradas antigas do heap são descartadas se sua distância não coincide com a melhor conhecida. Pesos zero são aceitos; negativos, NaN e infinitos são rejeitados. Indisponibilidade é representada por um conjunto de trechos bloqueados, preservando seu custo original para restauração.

**Justificativa:** com custos não negativos, uma continuação de caminho não reduz a distância já acumulada. Se houvesse caminho mais curto para o vértice mínimo retirado, sua primeira passagem pela fronteira dos vértices já processados teria oferecido uma distância menor no heap, contrariando essa escolha. A hipótese deixa de valer com pesos negativos.

Com listas de adjacência e heap binário, o custo usual para grafo simples é $O((|V|+|E|)\log|V|)$; a validação inicial percorre $O(|V|+|E|)$. O espaço é $O(|V|+|E|)$. Não há cache: toda consulta usa os pesos e as indisponibilidades atuais.

## 3. Interface e comportamento

| Operação | Comportamento |
|---|---|
| `calcular_menor_caminho(origem, destino)` | Retorna `(distancia, lista_de_equipamentos)` |
| `bloqueios={...}` | Exclui os equipamentos apenas nessa consulta |
| `arestas_bloqueadas={(u,v)}` | Exclui trechos apenas nessa consulta |
| `bloquear_trecho(u,v)` / `restaurar_trecho(u,v)` | Atualiza disponibilidade persistente do modelo |
| `atualizar_peso(u,v,valor)` | Atualiza custo finito; próxima consulta recalcula |
| Destino inalcançável ou terminal bloqueado | Retorna `(inf, [])` |
| Origem igual ao destino e disponível | Retorna `(0.0, [origem])` |
| Vértice/aresta desconhecido ou peso inválido | Levanta `ValueError` |

Não se converte erro de cadastro em “sem rota”: os dois resultados têm significados diferentes. Rotas empatadas podem ter mais de uma solução ótima; a implementação usa a ordem textual do heap para manter comportamento reproduzível. A estrutura é um grafo simples; conexões paralelas devem ser modeladas por nós intermediários ou por uma extensão explícita para multigrafos.

## 4. Exemplo resolvido e investigações

Da água até o dosador, a rota A custa $150+80+40=270$ cm; a B custa $150+85+40=275$ cm.

| Retirada válida do heap | Distância (cm) | Relaxação principal |
|---|---:|---|
| TK-101_Agua | 0 | Bombas A e B recebem 150 |
| P-101_BombaA | 150 | Manifold recebe 230 |
| P-102_BombaB | 150 | Candidato 235 não melhora 230 |
| MAN-200_Agua | 230 | Dosador recebe 270 |
| DOS-201_Dosador | 270 | Destino atingido |

1. **Traçado completo:** a tabela acima e `ultima_trilha` do notebook confirmam a distância nominal. Não há ciclo dirigido na bifurcação convergente das bombas; alternativas de caminho não exigem um ciclo dirigido.
2. **Peso −5:** o notebook rejeita essa entrada. Com uma aresta negativa, um vértice retirado antes pode ser melhorado depois; seria necessário outro algoritmo, como Bellman–Ford, e análise de ciclos negativos.
3. **Todos os pares:** repetir Dijkstra custa aproximadamente $O(|V|(|V|+|E|)\log|V|)$; Floyd–Warshall custa $O(|V|^3)$ e guarda $O(|V|^2)$ distâncias. Não existe um número universal de vértices de equilíbrio: a densidade $|E|$, a implementação e a quantidade de consultas também importam. Em grafos esparsos, Dijkstra repetido tende a ter melhor ordem assintótica; em densos, Floyd pode ser adequado.
4. **Bloqueio do manifold:** ambas as rotas se tornam inviáveis; o resultado é `(inf, [])`. A redundância das bombas não cobre o trecho comum até o dosador.
5. **Recálculo:** isolando a linha A, a rota passa por B com 275 cm. Isolando A e B, não há rota. Restaurando os trechos, voltamos a 270 cm. Alterando o comprimento A→manifold para 100 cm, A passa a custar 290 cm e B volta a ser ótima com 275 cm.

## 5. Validação executável

O notebook inclui verificações de rota completa e soma dos pesos, falha/restauração de trechos, bloqueio de nós, sentido das arestas, origem=destino, consulta sem mutação e rejeição de entradas inválidas. Um oráculo independente enumera caminhos simples de um grafo pequeno com ciclo, pesos zero e nó isolado: são **150 comparações** de distância em todos os pares, sem bloqueio e com cada bloqueio individual.

Esse oráculo serve somente aos testes; seu custo exponencial não é usado no roteamento. O arquivo possui saídas gravadas de execução local das células em ordem. Execute novamente para validar em seu ambiente; não execute com otimização `python -O`, que desativa `assert`.

## 6. Execução, limites e conclusão

Abra o notebook de mesmo nome no Jupyter ou Google Colab e execute todas as células em ordem, em Python 3.10 ou superior. Só são usados módulos da biblioteca padrão; nenhuma instalação, arquivo externo ou acesso à rede é necessário durante a execução.

As distâncias não representam perda de carga, vazão, tempo de atuação ou capacidade de bombeamento. Não há ensaio de hardware, comunicação com Automation Studio nem medição de latência de uma malha industrial. Um cálculo de rota não autoriza contornar dosagem, qualidade ou intertravamentos. Não foram alterados os arquivos 11 ou 15; este notebook é autônomo para preservar esse limite de escopo.

O entregável demonstra menor caminho, recálculo por peso e bloqueio, retorno de sequência/distância e tratamento explícito de falhas de cadastro e ausência de rota.

## 7. Referências e rastreabilidade

- [Roteiro da Aula 14 do professor](https://github.com/valerio-unifei/ECAA08-2026.2-Projeto/blob/c70e838aac076856727d0bb2151d20ff23d08600/etapa-02-grafos/14%20-%20Menor%20Caminho%20Dijkstra%20e%20Roteamento%20Dinamico.md).
- [Modelo do Grupo 2 — Aula 11](https://github.com/poggerssLL/Automatica---Grupo-2/blob/b25ada3f919adb3011a792fe62736cc941ac6c23/etapa-02-grafos/11%20-%20Modelagem%20da%20Tubulacao%20e%20Instrumentos%20como%20Grafo.md).
- [Modelo do Grupo 2 — Aula 15](https://github.com/poggerssLL/Automatica---Grupo-2/blob/b25ada3f919adb3011a792fe62736cc941ac6c23/etapa-02-grafos/15%20-%20Simulacao%20de%20Vazamentos%20e%20Desvio%20Automatico.md).


## Implementação autônoma

In [1]:
from math import inf, isfinite, isclose
from heapq import heappush, heappop
from numbers import Real


class GrafoDirigido:
    """Grafo simples dirigido: custos finitos não negativos e unidade única."""
    def __init__(self, vertices, unidade):
        vertices = tuple(vertices)
        if not vertices or any(not isinstance(v, str) or not v for v in vertices):
            raise ValueError("Informe vértices textuais não vazios")
        if len(set(vertices)) != len(vertices):
            raise ValueError("Vértices duplicados")
        if not isinstance(unidade, str) or not unidade:
            raise ValueError("Unidade obrigatória")
        self.unidade = unidade
        self.adj = {v: {} for v in vertices}
        self.indisponiveis = set()

    @property
    def vertices(self):
        return tuple(self.adj)

    def verificar_vertice(self, v):
        if v not in self.adj:
            raise ValueError(f"Vértice desconhecido: {v}")

    @staticmethod
    def verificar_peso(peso):
        if isinstance(peso, bool) or not isinstance(peso, Real):
            raise ValueError("Peso deve ser numérico")
        if not isfinite(peso) or peso < 0:
            raise ValueError("Dijkstra exige peso finito e não negativo")
        return float(peso)

    def adicionar_aresta(self, u, v, peso):
        self.verificar_vertice(u)
        self.verificar_vertice(v)
        peso = self.verificar_peso(peso)
        if v in self.adj[u]:
            raise ValueError("Aresta já cadastrada; use atualizar_peso")
        self.adj[u][v] = peso

    def verificar_aresta(self, u, v):
        self.verificar_vertice(u)
        self.verificar_vertice(v)
        if v not in self.adj[u]:
            raise ValueError(f"Aresta inexistente: {u} -> {v}")

    def atualizar_peso(self, u, v, peso):
        self.verificar_aresta(u, v)
        self.adj[u][v] = self.verificar_peso(peso)

    def bloquear_trecho(self, u, v):
        self.verificar_aresta(u, v)
        self.indisponiveis.add((u, v))

    def restaurar_trecho(self, u, v):
        self.verificar_aresta(u, v)
        self.indisponiveis.discard((u, v))

    def remover_aresta(self, u, v):
        self.verificar_aresta(u, v)
        del self.adj[u][v]
        self.indisponiveis.discard((u, v))


class DijkstraRouter:
    """Recalcula sobre o estado atual do grafo; (inf, []) significa sem rota."""
    def __init__(self, grafo):
        self.grafo = grafo
        self.ultima_trilha = []

    def calcular_menor_caminho(self, origem, destino, bloqueios=None,
                               arestas_bloqueadas=None):
        g = self.grafo
        g.verificar_vertice(origem)
        g.verificar_vertice(destino)
        bloqueios = set(bloqueios or ())
        arestas_bloqueadas = set(arestas_bloqueadas or ()) | g.indisponiveis
        for v in bloqueios:
            g.verificar_vertice(v)
        for u, v in arestas_bloqueadas:
            g.verificar_aresta(u, v)
        # Falha fechada mesmo se alguém editar adj diretamente.
        for u, vizinhos in g.adj.items():
            for v, peso in vizinhos.items():
                g.verificar_vertice(v)
                g.verificar_peso(peso)
        self.ultima_trilha = []
        if origem in bloqueios or destino in bloqueios:
            return inf, []
        dist = {v: inf for v in g.vertices}
        pred = {v: None for v in g.vertices}
        dist[origem] = 0.0
        heap = [(0.0, origem)]
        while heap:
            d, u = heappop(heap)
            if d != dist[u]:
                continue
            self.ultima_trilha.append((u, d))
            if u == destino:
                break
            for v, peso in g.adj[u].items():
                if v in bloqueios or (u, v) in arestas_bloqueadas:
                    continue
                candidato = d + peso
                if candidato < dist[v]:
                    dist[v], pred[v] = candidato, u
                    heappush(heap, (candidato, v))
        if dist[destino] == inf:
            return inf, []
        caminho, atual = [], destino
        while atual is not None:
            caminho.append(atual)
            atual = pred[atual]
        return dist[destino], list(reversed(caminho))


def validar_rota(g, resultado, origem, destino, bloqueios=(), arestas_bloqueadas=()):
    custo, rota = resultado
    if custo == inf:
        assert rota == []
        return
    assert rota[0] == origem and rota[-1] == destino
    assert len(rota) == len(set(rota))
    assert not set(rota).intersection(bloqueios)
    bloqueadas = set(arestas_bloqueadas) | g.indisponiveis
    soma = 0.0
    for u, v in zip(rota, rota[1:]):
        assert v in g.adj[u] and (u, v) not in bloqueadas
        soma += g.adj[u][v]
    assert isclose(soma, custo, rel_tol=1e-12, abs_tol=1e-12)


def deve_rejeitar(funcao, *args, **kwargs):
    try:
        funcao(*args, **kwargs)
    except ValueError:
        return
    raise AssertionError("Entrada inválida foi aceita")


def mostrar(nome, resultado, unidade):
    custo, rota = resultado
    if custo == inf:
        print(f"{nome}: SEM ROTA — nenhuma alternativa disponível")
    else:
        print(f"{nome}: {custo:.2f} {unidade} | " + " -> ".join(rota))


## Rede e cenários de recálculo

In [2]:
def criar_rede_envasadora():
    # Nomes e comprimentos preservados do modelo didático das Aulas 11/15.
    nos = ["TK-101_Agua", "P-101_BombaA", "P-102_BombaB", "MAN-200_Agua",
           "DOS-201_Dosador", "BIC-202_Bico", "MAG-100_Copos",
           "MESA-000_Index", "EST-500_Saida"]
    g = GrafoDirigido(nos, "cm")
    for u, v, peso in [
        ("TK-101_Agua", "P-101_BombaA", 150),
        ("TK-101_Agua", "P-102_BombaB", 150),
        ("P-101_BombaA", "MAN-200_Agua", 80),
        ("P-102_BombaB", "MAN-200_Agua", 85),
        ("MAN-200_Agua", "DOS-201_Dosador", 40),
        ("DOS-201_Dosador", "BIC-202_Bico", 20),
        ("BIC-202_Bico", "MESA-000_Index", 10),
        ("MAG-100_Copos", "MESA-000_Index", 15),
        ("MESA-000_Index", "EST-500_Saida", 50),
    ]:
        g.adicionar_aresta(u, v, peso)
    return g


rede = criar_rede_envasadora()
roteador = DijkstraRouter(rede)
ORIGEM, DESTINO = "TK-101_Agua", "DOS-201_Dosador"
nominal = roteador.calcular_menor_caminho(ORIGEM, DESTINO)
mostrar("Operação nominal", nominal, rede.unidade)
print("Retiradas válidas do heap:", roteador.ultima_trilha)
assert nominal == (270.0, [ORIGEM, "P-101_BombaA", "MAN-200_Agua", DESTINO])
validar_rota(rede, nominal, ORIGEM, DESTINO)

# O bloqueio é uma indisponibilidade lógica, sem perder o peso original.
rede.bloquear_trecho("P-101_BombaA", "MAN-200_Agua")
contingencia = roteador.calcular_menor_caminho(ORIGEM, DESTINO)
mostrar("Falha na linha A", contingencia, rede.unidade)
assert contingencia == (275.0, [ORIGEM, "P-102_BombaB", "MAN-200_Agua", DESTINO])
validar_rota(rede, contingencia, ORIGEM, DESTINO)
rede.bloquear_trecho("P-102_BombaB", "MAN-200_Agua")
assert roteador.calcular_menor_caminho(ORIGEM, DESTINO) == (inf, [])
mostrar("Falha simultânea A/B", (inf, []), rede.unidade)
rede.restaurar_trecho("P-101_BombaA", "MAN-200_Agua")
rede.restaurar_trecho("P-102_BombaB", "MAN-200_Agua")
assert roteador.calcular_menor_caminho(ORIGEM, DESTINO) == nominal

# Recálculo de peso finito: comparação por distância, não por perda de carga.
rede.atualizar_peso("P-101_BombaA", "MAN-200_Agua", 100)
reponderada = roteador.calcular_menor_caminho(ORIGEM, DESTINO)
mostrar("Trecho A alterado para 100 cm", reponderada, rede.unidade)
assert reponderada == contingencia
rede.atualizar_peso("P-101_BombaA", "MAN-200_Agua", 80)


Operação nominal: 270.00 cm | TK-101_Agua -> P-101_BombaA -> MAN-200_Agua -> DOS-201_Dosador
Retiradas válidas do heap: [('TK-101_Agua', 0.0), ('P-101_BombaA', 150.0), ('P-102_BombaB', 150.0), ('MAN-200_Agua', 230.0), ('DOS-201_Dosador', 270.0)]
Falha na linha A: 275.00 cm | TK-101_Agua -> P-102_BombaB -> MAN-200_Agua -> DOS-201_Dosador
Falha simultânea A/B: SEM ROTA — nenhuma alternativa disponível
Trecho A alterado para 100 cm: 275.00 cm | TK-101_Agua -> P-102_BombaB -> MAN-200_Agua -> DOS-201_Dosador


## Testes funcionais e oráculo independente

In [3]:
# Casos funcionais e de fronteira; consulta com bloqueios não altera o grafo.
g = criar_rede_envasadora()
r = DijkstraRouter(g)
for bloqueios, esperado in [
    ({"P-101_BombaA"}, 275.0),
    ({"MAN-200_Agua"}, inf),
    ({ORIGEM}, inf), ({DESTINO}, inf),
]:
    resultado = r.calcular_menor_caminho(ORIGEM, DESTINO, bloqueios)
    assert resultado[0] == esperado
    validar_rota(g, resultado, ORIGEM, DESTINO, bloqueios)
assert r.calcular_menor_caminho(ORIGEM, DESTINO)[0] == 270.0
assert r.calcular_menor_caminho(ORIGEM, ORIGEM) == (0.0, [ORIGEM])
assert r.calcular_menor_caminho(ORIGEM, ORIGEM, {ORIGEM}) == (inf, [])
assert r.calcular_menor_caminho(DESTINO, ORIGEM) == (inf, [])
assert r.calcular_menor_caminho("MAG-100_Copos", "EST-500_Saida")[0] == 65.0
assert r.calcular_menor_caminho(ORIGEM, DESTINO,
    arestas_bloqueadas={("P-101_BombaA", "MAN-200_Agua")})[0] == 275.0
for peso in (-5, float("nan"), inf, True, "80"):
    deve_rejeitar(g.atualizar_peso, "P-101_BombaA", "MAN-200_Agua", peso)
deve_rejeitar(r.calcular_menor_caminho, "INEXISTENTE", DESTINO)
deve_rejeitar(r.calcular_menor_caminho, ORIGEM, DESTINO, {"INEXISTENTE"})
deve_rejeitar(g.bloquear_trecho, ORIGEM, DESTINO)
deve_rejeitar(g.adicionar_aresta, ORIGEM, "P-101_BombaA", 150)

# Referência independente por enumeração de caminhos simples em grafo pequeno.
# Esse verificador exponencial é exclusivamente um oráculo de teste.
def referencia_exaustiva(g, origem, destino, bloqueios):
    if origem in bloqueios or destino in bloqueios:
        return inf
    def visitar(u, vistos, custo):
        if u == destino:
            return custo
        valores = [visitar(v, vistos | {v}, custo + w)
                   for v, w in g.adj[u].items()
                   if v not in vistos and v not in bloqueios]
        return min(valores, default=inf)
    return visitar(origem, {origem}, 0.0)

pequeno = GrafoDirigido(["A", "B", "C", "D", "E"], "cm")
for aresta in [("A", "B", 0), ("A", "C", 2), ("B", "C", 1),
               ("C", "B", 0), ("B", "D", 4), ("C", "D", 2)]:
    pequeno.adicionar_aresta(*aresta)
rp = DijkstraRouter(pequeno)
comparacoes = 0
for bloqueios in [set()] + [{v} for v in pequeno.vertices]:
    for origem in pequeno.vertices:
        for destino in pequeno.vertices:
            resultado = rp.calcular_menor_caminho(origem, destino, bloqueios)
            esperado = referencia_exaustiva(pequeno, origem, destino, bloqueios)
            assert resultado[0] == esperado
            validar_rota(pequeno, resultado, origem, destino, bloqueios)
            comparacoes += 1
print(f"TESTES 14 APROVADOS: casos funcionais/fronteira + {comparacoes} comparações independentes.")


TESTES 14 APROVADOS: casos funcionais/fronteira + 150 comparações independentes.
